# CLIP-Style Contrastive Learning — Training

In [ ]:
!pip install torch torchvision transformers kagglehub

## Download Dataset
Downloads Flickr8k from Kaggle via `kagglehub`. Requires a one-time Kaggle setup:
1. Go to kaggle.com → Settings → API → Create New Token
2. Upload the downloaded `kaggle.json` to `~/.kaggle/kaggle.json` on JupyterHub

In [ ]:
import os
import shutil
import kagglehub

DATASET_DIR = "dataset"
IMAGES_DIR = os.path.join(DATASET_DIR, "Images")
CAPTIONS_FILE = os.path.join(DATASET_DIR, "captions.txt")

if os.path.isdir(IMAGES_DIR) and os.path.isfile(CAPTIONS_FILE):
    print("Dataset already exists, skipping download.")
else:
    print("Downloading Flickr8k from Kaggle...")
    path = kagglehub.dataset_download("adityajn105/flickr8k")
    print(f"Downloaded to: {path}")

    os.makedirs(DATASET_DIR, exist_ok=True)

    src_images = os.path.join(path, "Images")
    src_captions = os.path.join(path, "captions.txt")

    if not os.path.isdir(IMAGES_DIR):
        shutil.copytree(src_images, IMAGES_DIR)
    if not os.path.isfile(CAPTIONS_FILE):
        shutil.copy2(src_captions, CAPTIONS_FILE)

    print(f"Dataset ready: {len(os.listdir(IMAGES_DIR))} images, captions.txt present.")

In [ ]:
import csv
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from transformers import BertModel, BertTokenizer

In [ ]:
# --------------- Config ---------------
CAPTIONS_FILE = "dataset/captions.txt"
IMAGES_DIR = "dataset/Images"
CHECKPOINT_DIR = "checkpoints"
EPOCHS = 10
LR = 1e-4
LOG_EVERY = 50
VAL_SPLIT = 0.1
BATCH_SIZE = 64
MAX_TOKEN_LENGTH = 64
EMBED_DIM = 256
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Device: {device}")

## Data Pipeline

In [ ]:
class Flickr8kDataset(Dataset):
    def __init__(self, captions_file: str, images_dir: str):
        self.images_dir = images_dir

        self.image_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

        self.tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

        self.samples = []  # list of (image_filename, caption)
        with open(captions_file, "r") as f:
            reader = csv.reader(f)
            next(reader)  # skip header
            for row in reader:
                image_filename = row[0]
                caption = row[1]
                self.samples.append((image_filename, caption))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_filename, caption = self.samples[idx]
        image_path = os.path.join(self.images_dir, image_filename)

        image = Image.open(image_path).convert("RGB")
        image = self.image_transform(image)

        tokens = self.tokenizer(
            caption,
            padding="max_length",
            truncation=True,
            max_length=MAX_TOKEN_LENGTH,
            return_tensors="pt",
        )
        input_ids = tokens["input_ids"].squeeze(0)
        attention_mask = tokens["attention_mask"].squeeze(0)

        return image, input_ids, attention_mask

In [ ]:
dataset = Flickr8kDataset(CAPTIONS_FILE, IMAGES_DIR)
val_size = int(len(dataset) * VAL_SPLIT)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)
print(f"Train: {train_size}  Val: {val_size}")

num_workers = 4 if torch.cuda.is_available() else 2
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=num_workers, pin_memory=True)

## Image Encoder

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        # Remove final classification layer — keep everything up to avgpool
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])  # output: (batch, 512, 1, 1)

        self.projection = nn.Sequential(
            nn.Linear(512, EMBED_DIM),
            nn.ReLU(),
            nn.Linear(EMBED_DIM, EMBED_DIM),
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        features = self.backbone(images).squeeze(-1).squeeze(-1)  # (batch, 512)
        projected = self.projection(features)                      # (batch, 256)
        return F.normalize(projected, dim=-1)                      # L2-normalized

## Text Encoder

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")

        self.projection = nn.Sequential(
            nn.Linear(768, EMBED_DIM),
            nn.ReLU(),
            nn.Linear(EMBED_DIM, EMBED_DIM),
        )

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] token — (batch, 768)
        projected = self.projection(cls_embedding)            # (batch, 256)
        return F.normalize(projected, dim=-1)                 # L2-normalized

## InfoNCE Loss

In [ ]:
class CLIPLoss(nn.Module):
    def __init__(self, initial_temperature: float = 0.07):
        super().__init__()
        # Learnable log-temperature: optimized during training alongside the encoders.
        # We store log(temperature) so that exp() always yields a positive scale factor.
        self.log_temperature = nn.Parameter(torch.tensor(initial_temperature).log())

    def forward(self, image_embeddings: torch.Tensor, text_embeddings: torch.Tensor) -> torch.Tensor:
        # Both inputs are L2-normalized, so matmul == cosine similarity
        # logits shape: (N, N)
        temperature = self.log_temperature.exp()
        logits = (image_embeddings @ text_embeddings.T) / temperature

        # Targets: the diagonal — pair i matches pair i
        N = logits.size(0)
        targets = torch.arange(N, device=logits.device)

        # Symmetric cross-entropy: image→text + text→image
        loss_i2t = F.cross_entropy(logits, targets)       # rows
        loss_t2i = F.cross_entropy(logits.T, targets)     # columns
        return (loss_i2t + loss_t2i) / 2

## Models & Optimizer

In [ ]:
image_encoder = ImageEncoder().to(device)
text_encoder = TextEncoder().to(device)
loss_fn = CLIPLoss().to(device)

optimizer = Adam(
    list(image_encoder.parameters()) + list(text_encoder.parameters()) + list(loss_fn.parameters()),
    lr=LR,
)

## Training Functions

In [ ]:
def train_one_epoch(image_encoder, text_encoder, loss_fn, optimizer, dataloader, device, epoch):
    image_encoder.train()
    text_encoder.train()

    running_loss = 0.0
    for step, (images, input_ids, attention_mask) in enumerate(dataloader):
        images = images.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        image_emb = image_encoder(images)
        text_emb = text_encoder(input_ids, attention_mask)
        loss = loss_fn(image_emb, text_emb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if (step + 1) % LOG_EVERY == 0:
            avg = running_loss / LOG_EVERY
            temp = loss_fn.log_temperature.exp().item()
            print(f"  Epoch {epoch+1} | Step {step+1:>4d} | Loss {avg:.4f} | Temp {temp:.4f}")
            running_loss = 0.0

    return running_loss


@torch.no_grad()
def validate(image_encoder, text_encoder, loss_fn, dataloader, device):
    image_encoder.eval()
    text_encoder.eval()

    total_loss = 0.0
    num_batches = 0
    for images, input_ids, attention_mask in dataloader:
        images = images.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        image_emb = image_encoder(images)
        text_emb = text_encoder(input_ids, attention_mask)
        loss = loss_fn(image_emb, text_emb)

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches

## Training Loop

In [ ]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
best_val_loss = float("inf")

for epoch in range(EPOCHS):
    t0 = time.time()
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"{'='*50}")

    train_one_epoch(image_encoder, text_encoder, loss_fn, optimizer, train_loader, device, epoch)

    val_loss = validate(image_encoder, text_encoder, loss_fn, val_loader, device)
    elapsed = time.time() - t0
    print(f"  Val Loss: {val_loss:.4f} | Time: {elapsed:.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint = {
            "epoch": epoch + 1,
            "val_loss": val_loss,
            "image_encoder": image_encoder.state_dict(),
            "text_encoder": text_encoder.state_dict(),
            "loss_fn": loss_fn.state_dict(),
            "optimizer": optimizer.state_dict(),
        }
        path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
        torch.save(checkpoint, path)
        print(f"  ** New best model saved (val_loss={val_loss:.4f}) **")

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")